In [ ]:
import pandas as pd

input_filename = '数据处理.csv'
output_filename = 'firm_monthly_panel_data_revised.csv' 


# --- 第1步：加载原始数据 ---
print(f"--- 第1步：正在从 '{input_filename}' 加载数据... ---")
try:
    
    df_original = pd.read_csv(input_filename, encoding='utf-8')
    print(f"成功加载数据 (UTF-8 编码)。")
except FileNotFoundError:
    print(f"错误：找不到文件 '{input_filename}'。请确保该文件与您的Python脚本在同一个目录下。")
    exit() 
except Exception as e:
    print(f"使用 UTF-8 编码加载文件失败: {e}")
    print("正在尝试使用 'gbk' 编码...")
    try:
        
        df_original = pd.read_csv(input_filename, encoding='gbk')
        print(f"成功加载数据 (GBK 编码)。")
    except Exception as e2:
        print(f"使用 'gbk' 编码也失败了: {e2}")
        exit() 

print("原始数据前5行预览:")
print(df_original.head())
print("\n")


# --- 第2步：重命名列为英文 ---
print("--- 第2步：正在重命名列为英文... ---")
column_mapping = {
    '岗位名称': 'job_title', 'Glassdoor公司链接': 'glassdoor_company_link', '公司名称': 'company_name',
    '公司代码': 'company_code', '评论日期': 'review_date', '评分': 'overall_rating',
    '评论标题': 'review_title', '评论者状态': 'reviewer_status', '正评': 'pros',
    '负评': 'cons', '建议': 'advice_to_management', '推荐公司': 'recommends',
    'CEO支持率': 'ceo_approval', '公司展望': 'business_outlook', '职业机会': 'career_opportunities',
    '薪酬与福利': 'compensation_benefits', '高级管理层': 'senior_management', '工作与生活平衡': 'work_life_balance',
    '文化与价值观': 'culture_values', '多元化与包容性': 'diversity_inclusion', '索引编号': 'review_id'
}
df_eng = df_original.rename(columns=column_mapping)
print("重命名完成。数据预览:")
print(df_eng.head())
print("\n")


# --- 第3步：处理日期并创建聚合键 ---
print("--- 第3步：正在处理日期并创建 'year_month' 键... ---")
df_eng['review_date'] = pd.to_datetime(df_eng['review_date'], errors='coerce')
initial_rows = len(df_eng)
df_eng.dropna(subset=['review_date'], inplace=True) # 移除日期格式不正确的行
rows_dropped = initial_rows - len(df_eng)
if rows_dropped > 0:
    print(f"移除了 {rows_dropped} 行因日期格式无效。")
df_eng['year_month'] = df_eng['review_date'].dt.to_period('M')
print("日期处理完成。")
print(df_eng[['company_code', 'review_date', 'year_month']].head())
print("\n")


# --- 第4步：确保评分类是数值型 ---
print("--- 第4步：正在转换评分为数值型... ---")
rating_columns = [
    'overall_rating', 'career_opportunities', 'compensation_benefits',
    'senior_management', 'work_life_balance', 'culture_values',
    'diversity_inclusion'
]
for col in rating_columns:
    df_eng[col] = pd.to_numeric(df_eng[col], errors='coerce')
print("转换完成。检查数据类型:")
print(df_eng[rating_columns].dtypes)
print("\n")


# --- 第5步：为稳健的分组创建公司名称映射 ---
print("--- 第5步：创建公司代码到名称的映射... ---")
company_name_map = df_eng[['company_code', 'company_name']].drop_duplicates().set_index('company_code')
print(f"成功创建了 {len(company_name_map)} 个独立公司的映射。")
print(company_name_map.head())
print("\n")


# --- 第6步：执行核心聚合操作  ---
print("--- 第6步：使用命名聚合生成面板数据... ---")
df_panel = df_eng.groupby(['company_code', 'year_month']).agg(
    
    monthly_rating_count=('review_id', 'count'),
    compensation_benefits_mean=('compensation_benefits', 'mean'),
    compensation_benefits_dispersion=('compensation_benefits', 'std'),
    work_life_balance_mean=('work_life_balance', 'mean'),
    
    
    overall_rating_mean=('overall_rating', 'mean'),
    career_opportunities_mean=('career_opportunities', 'mean'),
    senior_management_mean=('senior_management', 'mean'),
    culture_values_mean=('culture_values', 'mean'),
    diversity_inclusion_mean=('diversity_inclusion', 'mean')
).reset_index()
print("聚合完成。")
print("\n")


# --- 第7步：将公司名称合并回面板数据 ---

print("--- 第7步：将公司名称合并回结果... ---")
df_panel = pd.merge(df_panel, company_name_map, on='company_code', how='left')

company_name_col = df_panel.pop('company_name')
df_panel.insert(1, 'company_name', company_name_col) 

print("最终面板数据预览:")
print(df_panel.head())
print("\n")


# --- 第8步：保存结果到新文件 ---
print(f"--- 第8步：正在保存结果到 '{output_filename}'... ---")
try:
    df_panel.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"--- 成功！处理完成的数据已保存到文件：'{output_filename}' ---")
except Exception as e:
    print(f"保存文件时出错: {e}")

--- 第1步：正在从 '数据处理.csv' 加载数据... ---


C:\Users\32678\AppData\Local\Temp\ipykernel_30396\1520224432.py:16: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_original = pd.read_csv(input_filename, encoding='utf-8')


成功加载数据 (UTF-8 编码)。
原始数据前5行预览:
                        岗位名称  \
0             Manager Design   
1         Anonymous Employee   
2        Production Engineer   
3   Senior Account Executive   
4          Assistant Manager   

                                       Glassdoor公司链接                  公司名称  \
0  Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm  Baja Steel and Fence   
1  Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm  Baja Steel and Fence   
2  Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm  Baja Steel and Fence   
3  https://www.glassdoor.com/Reviews/Calgary-Flam...        Calgary Flames   
4  https://www.glassdoor.com/Reviews/Calgary-Flam...        Calgary Flames   

      公司代码        评论日期   评分  \
0  5462645  2022-11-19  5.0   
1  5462645  2022-01-29  4.0   
2  5462645  2021-08-12  4.0   
3     5247  2020-09-24  1.0   
4     5247  2023-03-25  4.0   

                                                评论标题  \
0                                               Good   
1        